In [ ]:
# =========================
# Google Colab: Lovable -> HTTPS Backend -> OpenAI (stateless)
# One-cell run: installs deps, starts FastAPI, opens public HTTPS URL, prints it.
# =========================

import os, sys, re, json, time, asyncio, threading, subprocess, urllib.request, textwrap
from typing import Any, Dict, List, Optional, Literal

# ---------- Install dependencies ----------
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "fastapi>=0.110.0",
                       "uvicorn[standard]>=0.27.0",
                       "openai>=1.40.0",
                       "pydantic>=2.0.0"])

# ---------- Imports after install ----------
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from openai import OpenAI

# ---------- Settings ----------
HOST = "0.0.0.0"
PORT = int(os.environ.get("PORT", "8000"))
DEFAULT_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")  # you can override via env: OPENAI_MODEL
MAX_OUTPUT_TOKENS = int(os.environ.get("MAX_OUTPUT_TOKENS", "900"))

# ---------- Colab Secrets: recommended way ----------
OPENAI_API_KEY = None
try:
    from google.colab import userdata  # type: ignore
    OPENAI_API_KEY = userdata.get("UI_OPENAI_KEY")
except Exception:
    OPENAI_API_KEY = os.environ.get("UI_OPENAI_KEY")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# ---------- Prompt ----------
SYSTEM_PROMPT = textwrap.dedent("""
Ты — креативный сценарист и редактор. Ты получаешь:
1) РЕЖИМ: "book" (сценарий книги) или "video" (сценарий видеоролика)
2) ИСТОРИЮ ДИАЛОГА целиком
3) ТЕКУЩИЙ СЦЕНАРИЙ из 3 блоков: Вступление, Развитие, Финал (могут быть пустыми)
4) НОВОЕ СООБЩЕНИЕ пользователя

Правила поведения:
- Сначала определи намерение пользователя:
  A) ВОПРОС (пояснение, консультация, уточнение, без прямой просьбы создать/изменить сценарий)
  B) ЗАПРОС НА СЦЕНАРИЙ (создать сценарий с нуля, заполнить, переписать, улучшить, добавить/удалить элементы, изменить стиль, структуру, героев, темп, длительность и т.п.)

- Если это ВОПРОС:
  * Дай понятный ответ в чат.
  * СЦЕНАРИЙ НЕ МЕНЯЙ НИ ПРИ КАКИХ УСЛОВИЯХ: верни блоки ровно такими, как пришли.

- Если это ЗАПРОС НА СЦЕНАРИЙ:
  * Верни обновлённый сценарий из 3 блоков.
  * Если сценарий пустой и у пользователя общая идея — сразу заполни все 3 блока полностью (черновик целиком).
  * Если сценарий уже есть — меняй только те части, которые реально нужно изменить по смыслу запроса. Остальное сохраняй.
  * Если пользователь НЕ просит правки сценария явно — изменения запрещены.

- Всегда верни JSON:
  1) chat — текст ответа ассистента для чата (что ты сделал и кратко почему)
  2) scenario — три блока (вступление, развитие, финал) — ВСЕГДА, даже если они не менялись
  3) kind — "qa" если это ответ на вопрос без изменения сценария, иначе "scenario_update"

Вывод должен строго соответствовать заданной JSON-схеме. Никаких лишних ключей, текста, пояснений, markdown или обрамления.
""").strip()

# ---------- Request / Response Schemas ----------
class ScenarioBlocks(BaseModel):
    intro: str = Field(default="")
    development: str = Field(default="")
    final: str = Field(default="")

class FrontRequest(BaseModel):
    mode: Literal["book", "video"] = Field(..., description="Selected mode: book or video")
    history: Optional[Any] = Field(default=None, description="Full dialog history (string or array of messages)")
    scenario: Optional[Dict[str, Any]] = Field(default=None, description="Current scenario blocks")
    user_message: str = Field(..., description="New user message")

class ModelResult(BaseModel):
    kind: Literal["qa", "scenario_update"]
    chat: str
    scenario: ScenarioBlocks

def _coerce_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)

def normalize_scenario(s: Optional[Dict[str, Any]]) -> ScenarioBlocks:
    if not isinstance(s, dict):
        return ScenarioBlocks(intro="", development="", final="")
    # Accept common aliases (ru/en)
    intro = s.get("intro", s.get("introduction", s.get("вступление", s.get("start", ""))))
    dev = s.get("development", s.get("middle", s.get("развитие", s.get("body", ""))))
    fin = s.get("final", s.get("finale", s.get("ending", s.get("финал", s.get("end", "")))))
    return ScenarioBlocks(intro=_coerce_str(intro), development=_coerce_str(dev), final=_coerce_str(fin))

def normalize_history(h: Any) -> List[Dict[str, str]]:
    """
    Accepts:
      - list of {role, content}
      - list of strings
      - a single string
    Returns list of OpenAI-style messages (role: system/user/assistant).
    """
    msgs: List[Dict[str, str]] = []
    if h is None:
        return msgs
    if isinstance(h, str):
        txt = h.strip()
        if txt:
            msgs.append({"role": "user", "content": f"[HISTORY_AS_TEXT]\n{txt}"})
        return msgs
    if isinstance(h, list):
        for item in h:
            if isinstance(item, dict):
                role = str(item.get("role", "user")).strip().lower()
                if role not in ("system", "user", "assistant", "developer"):
                    role = "user"
                content = _coerce_str(item.get("content", ""))
                if content.strip():
                    msgs.append({"role": role, "content": content})
            elif isinstance(item, str):
                if item.strip():
                    msgs.append({"role": "user", "content": item})
            else:
                s = _coerce_str(item)
                if s.strip():
                    msgs.append({"role": "user", "content": s})
        return msgs
    # Fallback
    txt = _coerce_str(h).strip()
    if txt:
        msgs.append({"role": "user", "content": f"[HISTORY_AS_TEXT]\n{txt}"})
    return msgs

def build_context_message(mode: str, scenario: ScenarioBlocks, user_message: str) -> str:
    return textwrap.dedent(f"""
    [MODE]
    {mode}

    [CURRENT_SCENARIO]
    [INTRO]
    {scenario.intro}

    [DEVELOPMENT]
    {scenario.development}

    [FINAL]
    {scenario.final}

    [NEW_USER_MESSAGE]
    {user_message}

    [TASK]
    Определи намерение: "qa" (вопрос) или "scenario_update" (создать/изменить сценарий).
    Верни JSON строго по схеме.
    """).strip()

async def call_openai_structured(messages: List[Dict[str, str]], model: str) -> ModelResult:
    # Run sync OpenAI client call in a thread to avoid blocking event loop.
    def _sync_call() -> ModelResult:
        if client is None:
            raise RuntimeError("OPENAI_API_KEY is missing")
        resp = client.responses.parse(
            model=model,
            input=messages,
            text_format=ModelResult,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            store=False
        )
        parsed = getattr(resp, "output_parsed", None)
        if parsed is None:
            raise RuntimeError("Model returned no parsed output")
        return parsed

    return await asyncio.to_thread(_sync_call)

# ---------- FastAPI app ----------
app = FastAPI(title="Lovable GPT Backend (Stateless)", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # for browser calls
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def root():
    return {"ok": True, "service": "lovable-gpt-backend", "version": "1.0.0"}

@app.post("/v1/generate")
async def generate(req: FrontRequest, request: Request):
    # Key check
    if not OPENAI_API_KEY or client is None:
        return JSONResponse(
            status_code=401,
            content={
                "ok": False,
                "error": "OPENAI_API_KEY not found in Colab Secrets. Add it in Colab (Secrets) with name OPENAI_API_KEY. Без ключа работать не может.",
                "assistant_message": "Я не могу обратиться к модели без OPENAI_API_KEY. Добавьте ключ в Colab Secrets (имя: OPENAI_API_KEY).",
                "apply_scenario": False,
                "kind": "qa",
                "scenario": normalize_scenario(req.scenario).model_dump()
            },
        )

    mode = req.mode
    scenario_in = normalize_scenario(req.scenario)
    user_message = (req.user_message or "").strip()

    # Build OpenAI input
    history_msgs = normalize_history(req.history)

    # Avoid accidental duplication if history already ends with same user message
    if history_msgs:
        last = history_msgs[-1]
        if last.get("role") == "user" and (last.get("content") or "").strip() == user_message:
            history_msgs = history_msgs[:-1]

    context_msg = build_context_message(mode, scenario_in, user_message)

    messages: List[Dict[str, str]] = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history_msgs)
    messages.append({"role": "user", "content": context_msg})

    # Call model with robust fallback behavior
    result: Optional[ModelResult] = None
    used_model = DEFAULT_MODEL

    try:
        result = await call_openai_structured(messages, used_model)
    except Exception:
        # Fallback to a broadly available model name if DEFAULT_MODEL is unavailable
        try:
            used_model = "gpt-4o-mini"
            result = await call_openai_structured(messages, used_model)
        except Exception:
            # Safe fallback: unchanged scenario, helpful message
            return JSONResponse(
                status_code=200,
                content={
                    "ok": True,
                    "kind": "qa",
                    "assistant_message": "Не удалось получить корректный структурированный ответ от модели. Я не меняю сценарий. Попробуйте переформулировать запрос.",
                    "apply_scenario": False,
                    "scenario": scenario_in.model_dump(),
                    "meta": {"model": used_model, "fallback": True},
                },
            )

    # Enforce rule: if QA, scenario MUST remain exactly as input
    if result.kind == "qa":
        scenario_out = scenario_in
        apply_scenario = False
        kind = "qa"
    else:
        scenario_out = result.scenario or scenario_in
        apply_scenario = True
        kind = "scenario_update"

    return {
        "ok": True,
        "kind": kind,
        "assistant_message": (result.chat or "").strip(),
        "apply_scenario": apply_scenario,
        "scenario": scenario_out.model_dump(),
        "meta": {"model": used_model}
    }

# ---------- Start Uvicorn in background thread ----------
def run_server():
    import uvicorn
    uvicorn.run(app, host=HOST, port=PORT, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(1.5)

# ---------- Download and run cloudflared quick tunnel (HTTPS) ----------
CLOUDFLARED_PATH = "/content/cloudflared"
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

def ensure_cloudflared():
    if not os.path.exists(CLOUDFLARED_PATH):
        urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
        os.chmod(CLOUDFLARED_PATH, 0o755)

ensure_cloudflared()

# Start tunnel and parse public URL
cmd = [CLOUDFLARED_PATH, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
url_re = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")

start_time = time.time()
while time.time() - start_time < 30:
    line = proc.stdout.readline() if proc.stdout else ""
    if not line:
        if proc.poll() is not None:
            break
        time.sleep(0.1)
        continue
    m = url_re.search(line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print("❌ Не удалось получить публичный URL от cloudflared. Логи cloudflared продолжаются ниже:\n")
    try:
        for _ in range(50):
            line = proc.stdout.readline() if proc.stdout else ""
            if not line:
                break
            print(line.rstrip())
    except Exception:
        pass
    print("\nПроверьте, что cloudflared может подключиться к сети из Colab.")
else:
    print("\n==================== PUBLIC HTTPS URL ====================")
    print(public_url)
    print("POST endpoint:", public_url + "/v1/generate")
    print("Health check :", public_url + "/")
    print("==========================================================\n")

# ---------- Keep cell running ----------
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    pass

INFO:     Started server process [954]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



==================== PUBLIC HTTPS URL ====================
https://manufacturers-penalties-gotta-issue.trycloudflare.com
POST endpoint: https://manufacturers-penalties-gotta-issue.trycloudflare.com/v1/generate
Health check : https://manufacturers-penalties-gotta-issue.trycloudflare.com/

INFO:     66.94.103.157:0 - "OPTIONS /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     209.50.48.26:0 - "OPTIONS /v1/generate HTTP/1.1" 200 OK
INFO:     209.50.48.26:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     209.50.48.26:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     66.94.103.157:0 - "POST /v1/generate HTTP/1.1" 200 OK
